# GR00T N1.7 — CKA pruning, recovery fine-tuning và đánh giá đầy đủ

Notebook chạy theo thứ tự: cài đặt → access token ẩn → CKA calibration → sinh pruning manifest → fine-tune baseline/CKA → benchmark → heatmap trước/sau recovery → kết luận tự động.

> Mặc định `EXPERIMENT_MODE='smoke'` để kiểm tra pipeline trên T4. Kết quả nghiên cứu cần `full`, dữ liệu train/calibration/validation tách biệt và GPU tối thiểu khoảng 48 GiB. Chọn **Run all** sau khi sửa cell cấu hình.

In [ ]:
from pathlib import Path
import os

# ===== Chỉ sửa cell này =====
REPO_URL = 'https://github.com/<YOUR_USER>/Isaac-GR00T.git'
BRANCH = 'cka-n1d7-recovery'
MODEL_PATH = 'nvidia/GR00T-N1.7-3B'
EMBODIMENT_TAG = 'OXE_DROID_RELATIVE_EEF_RELATIVE_JOINT'

EXPERIMENT_MODE = 'smoke'  # 'smoke' cho T4; 'full' cho recovery thực nghiệm
RUN_TRAINING = True
RUN_BENCHMARK = True
RUN_HEATMAPS = True

IS_KAGGLE = Path('/kaggle/working').exists()
WORK_ROOT = Path('/kaggle/working' if IS_KAGGLE else '/content')
REPO_DIR = WORK_ROOT / 'Isaac-GR00T'
OUTPUT_ROOT = WORK_ROOT / 'gr00t_n1d7_cka'

# Có thể dùng đường dẫn tuyệt đối. Đường dẫn tương đối được tính từ repo.
CALIBRATION_DATASET = 'demo_data/droid_sample'
TRAIN_DATASET = 'demo_data/droid_sample'
VALIDATION_DATASET = 'demo_data/droid_sample'
DATASETS_ARE_DISJOINT = False  # đặt True khi dùng train/calibration/validation thật

CALIBRATION_TRAJECTORIES = [0]
VALIDATION_TRAJECTORIES = [0]
SAMPLES_PER_CALIBRATION_TRAJECTORY = 16
SAMPLES_PER_VALIDATION_TRAJECTORY = 40
SAMPLE_STRIDE = 4
DENOISING_STEPS = 4
SEED = 42

# Dùng checkpoint có sẵn bằng cách đặt RUN_TRAINING=False và điền hai đường dẫn.
BASELINE_CHECKPOINT = ''
CKA_CHECKPOINT = ''

assert EXPERIMENT_MODE in {'smoke', 'full'}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['MPLBACKEND'] = 'Agg'
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['GR00T_LOW_VRAM_T4'] = '1'


In [ ]:
# Helper: thành công thì không in log; thất bại in toàn bộ chi tiết và dừng.
import re
import subprocess
import sys

LOG_DIR = OUTPUT_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run_checked(name, command, cwd=None, env=None):
    safe_name = re.sub(r'[^A-Za-z0-9_.-]+', '_', name).strip('_')
    log_path = LOG_DIR / f'{safe_name}.log'
    completed = subprocess.run(
        [str(item) for item in command],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    log_path.write_text(completed.stdout or '', encoding='utf-8')
    if completed.returncode != 0:
        print(f'===== LỖI: {name} =====')
        print('Command:', ' '.join(map(str, command)))
        print('Log:', log_path)
        print(completed.stdout)
        raise RuntimeError(f'{name} failed with exit code {completed.returncode}')
    return log_path

def uv_python(script, *arguments):
    return ['uv', 'run', '--no-sync', 'python', script, *map(str, arguments)]

def resolve_repo_path(value):
    path = Path(value)
    return path if path.is_absolute() else REPO_DIR / path


In [ ]:
# Kiểm tra phần cứng; cell này chủ động hiển thị thông tin.
for command in (['nvidia-smi'], ['bash', '-lc', 'free -h'], ['bash', '-lc', 'df -h /']):
    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stdout if result.returncode == 0 else result.stderr)


In [ ]:
# Clone đúng branch và cài môi trường.
run_checked('Install uv and huggingface_hub', [
    sys.executable, '-m', 'pip', 'install', '-q', 'uv', 'huggingface_hub'
])
if not REPO_DIR.exists():
    if '<YOUR_USER>' in REPO_URL:
        raise ValueError('Hãy sửa REPO_URL trong cell cấu hình trước khi Run all')
    run_checked('Clone repository', [
        'git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR
    ])
else:
    run_checked('Verify branch', ['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run_checked('Update branch', ['git', 'pull', '--ff-only'], cwd=REPO_DIR)
run_checked('Install GR00T environment', ['uv', 'sync', '--all-extras'], cwd=REPO_DIR)


In [ ]:
# Nhập Hugging Face access token dưới dạng ẩn. Token không được ghi vào notebook/log.
from getpass import getpass
from huggingface_hub import login, whoami

hf_token = getpass('Hugging Face read token: ')
login(token=hf_token, add_to_git_credential=False)
del hf_token
print('Đã xác thực Hugging Face:', whoami()['name'])


In [ ]:
# Kiểm tra dependency/GPU trong đúng .venv của repo.
run_checked('Verify PyTorch CUDA', uv_python('-c', (
    "import torch; assert torch.cuda.is_available(); "
    "print(torch.__version__, torch.cuda.get_device_name(0))"
)), cwd=REPO_DIR, env=os.environ.copy())
for dataset in (CALIBRATION_DATASET, TRAIN_DATASET, VALIDATION_DATASET):
    if not resolve_repo_path(dataset).exists():
        raise FileNotFoundError(f'Dataset không tồn tại: {resolve_repo_path(dataset)}')


## A. Thu activation từ pretrained N1.7 và tính CKA
Các keep-index được tính từ activation N1.7 hiện tại; notebook không sử dụng keep-list hardcode của N1.5.

In [ ]:
SOURCE_CALIBRATION_DIR = OUTPUT_ROOT / 'source_calibration'
ANALYSIS_DIR = OUTPUT_ROOT / 'source_cka_analysis'
capture_common = [
    '--dataset-path', resolve_repo_path(CALIBRATION_DATASET),
    '--embodiment-tag', EMBODIMENT_TAG,
    '--trajectory-ids', *CALIBRATION_TRAJECTORIES,
    '--samples-per-trajectory', SAMPLES_PER_CALIBRATION_TRAJECTORY,
    '--sample-stride', SAMPLE_STRIDE,
    '--denoising-steps', 1,
    '--seed', SEED,
    '--require-hf-token',
]
run_checked('Capture source activations', uv_python(
    'scripts/cka_n1d7/capture_activations.py',
    '--model-path', MODEL_PATH,
    '--output-dir', SOURCE_CALIBRATION_DIR,
    *capture_common,
), cwd=REPO_DIR, env=os.environ.copy())
run_checked('Analyze source CKA', uv_python(
    'scripts/cka_n1d7/analyze_cka.py',
    '--calibration-dir', SOURCE_CALIBRATION_DIR,
    '--output-dir', ANALYSIS_DIR,
    '--backbone-language-prune-ratio', 0.40,
    '--action-dit-prune-ratio', 0.50,
    '--vl-self-attention-prune-ratio', 0.25,
), cwd=REPO_DIR, env=os.environ.copy())
PRUNING_MANIFEST = ANALYSIS_DIR / 'pruning_manifest.json'


## B. Recovery fine-tuning baseline và CKA
Hai nhánh dùng cùng base checkpoint, dữ liệu, seed, batch, learning rate và số bước. Biến duy nhất là pruning manifest. `smoke` chỉ chứng minh pipeline hoạt động, không phải kết quả nghiên cứu.

In [ ]:
BASELINE_OUTPUT = OUTPUT_ROOT / 'baseline_recovery'
CKA_OUTPUT = OUTPUT_ROOT / 'cka_recovery'
MAX_STEPS = 20 if EXPERIMENT_MODE == 'smoke' else 10_000
SAVE_STEPS = MAX_STEPS if EXPERIMENT_MODE == 'smoke' else 1_000
GLOBAL_BATCH = 1 if EXPERIMENT_MODE == 'smoke' else 8
GRADIENT_ACCUMULATION = 4 if EXPERIMENT_MODE == 'smoke' else 8

def recovery_command(output_dir, manifest=None):
    command = uv_python(
        'gr00t/experiment/launch_finetune.py',
        '--base-model-path', MODEL_PATH,
        '--dataset-path', resolve_repo_path(TRAIN_DATASET),
        '--embodiment-tag', EMBODIMENT_TAG,
        '--output-dir', output_dir,
        '--global-batch-size', GLOBAL_BATCH,
        '--gradient-accumulation-steps', GRADIENT_ACCUMULATION,
        '--learning-rate', 1e-4,
        '--max-steps', MAX_STEPS,
        '--save-steps', SAVE_STEPS,
    )
    if manifest is not None:
        command += ['--cka-pruning-manifest-path', manifest]
    if EXPERIMENT_MODE == 'smoke':
        command += ['--no-tune-diffusion-model', '--no-tune-vlln']
    return command

if RUN_TRAINING:
    run_checked('Fine-tune baseline', recovery_command(BASELINE_OUTPUT),
                cwd=REPO_DIR, env=os.environ.copy())
    run_checked('Fine-tune CKA recovery', recovery_command(CKA_OUTPUT, PRUNING_MANIFEST),
                cwd=REPO_DIR, env=os.environ.copy())
    BASELINE_CHECKPOINT = str(BASELINE_OUTPUT)
    CKA_CHECKPOINT = str(CKA_OUTPUT)
elif not BASELINE_CHECKPOINT or not CKA_CHECKPOINT:
    raise ValueError('RUN_TRAINING=False yêu cầu BASELINE_CHECKPOINT và CKA_CHECKPOINT')


## C. Benchmark công bằng
Đo parameter count, training runtime, mean/median/P95 latency, peak VRAM, MSE/MAE và predicted-vs-ground-truth trên cùng validation samples.

In [ ]:
EVAL_BASELINE = OUTPUT_ROOT / 'eval_baseline'
EVAL_CKA = OUTPUT_ROOT / 'eval_cka'
FINAL_COMPARISON = OUTPUT_ROOT / 'final_comparison'

def benchmark_command(model_path, run_name, output_dir):
    return uv_python(
        'scripts/cka_n1d7/benchmark_finetuned.py',
        '--model-path', model_path,
        '--dataset-path', resolve_repo_path(VALIDATION_DATASET),
        '--embodiment-tag', EMBODIMENT_TAG,
        '--output-dir', output_dir,
        '--run-name', run_name,
        '--trajectory-ids', *VALIDATION_TRAJECTORIES,
        '--samples-per-trajectory', SAMPLES_PER_VALIDATION_TRAJECTORY,
        '--sample-stride', SAMPLE_STRIDE,
        '--warmup-steps', 5,
        '--denoising-steps', DENOISING_STEPS,
        '--seed', SEED,
    )

if RUN_BENCHMARK:
    run_checked('Benchmark fine-tuned baseline',
                benchmark_command(BASELINE_CHECKPOINT, 'baseline-finetuned', EVAL_BASELINE),
                cwd=REPO_DIR, env=os.environ.copy())
    run_checked('Benchmark recovered CKA',
                benchmark_command(CKA_CHECKPOINT, 'cka-recovered', EVAL_CKA),
                cwd=REPO_DIR, env=os.environ.copy())
    run_checked('Compare fine-tuned checkpoints', uv_python(
        'scripts/cka_n1d7/compare_finetuned.py',
        '--baseline-dir', EVAL_BASELINE,
        '--cka-dir', EVAL_CKA,
        '--output-dir', FINAL_COMPARISON,
    ), cwd=REPO_DIR, env=os.environ.copy())


## D. Heatmap CKA baseline fine-tuned và CKA-pruned + recovery
Thu lại activation trên chính các calibration observations/seeds đã dùng. Heatmap delta căn theo original layer index, không căn theo vị trí ModuleList mới.

In [ ]:
BASELINE_HEATMAP_CAL = OUTPUT_ROOT / 'baseline_heatmap_calibration'
CKA_HEATMAP_CAL = OUTPUT_ROOT / 'cka_heatmap_calibration'
HEATMAP_COMPARISON = OUTPUT_ROOT / 'heatmap_comparison'
if RUN_HEATMAPS:
    run_checked('Capture baseline recovered activations', uv_python(
        'scripts/cka_n1d7/capture_activations.py',
        '--model-path', BASELINE_CHECKPOINT,
        '--output-dir', BASELINE_HEATMAP_CAL,
        *capture_common,
    ), cwd=REPO_DIR, env=os.environ.copy())
    run_checked('Capture CKA recovered activations', uv_python(
        'scripts/cka_n1d7/capture_activations.py',
        '--model-path', CKA_CHECKPOINT,
        '--output-dir', CKA_HEATMAP_CAL,
        *capture_common,
    ), cwd=REPO_DIR, env=os.environ.copy())
    run_checked('Compare recovered CKA heatmaps', uv_python(
        'scripts/cka_n1d7/compare_cka_heatmaps.py',
        '--baseline-calibration-dir', BASELINE_HEATMAP_CAL,
        '--optimized-calibration-dir', CKA_HEATMAP_CAL,
        '--output-dir', HEATMAP_COMPARISON,
    ), cwd=REPO_DIR, env=os.environ.copy())


In [ ]:
# Hiển thị biểu đồ và heatmap.
from IPython.display import Image, display

images = []
if FINAL_COMPARISON.exists():
    images.append(FINAL_COMPARISON / 'normalized_comparison.png')
if EVAL_BASELINE.exists():
    images.append(EVAL_BASELINE / 'prediction_vs_ground_truth.png')
if EVAL_CKA.exists():
    images.append(EVAL_CKA / 'prediction_vs_ground_truth.png')
if HEATMAP_COMPARISON.exists():
    images.extend(sorted(HEATMAP_COMPARISON.glob('cka_before_after_*.png')))
for image_path in images:
    if image_path.exists():
        display(Image(filename=str(image_path)))


In [ ]:
# Kết luận tự động, luôn gắn nhãn smoke/full và độ hợp lệ của dữ liệu.
import json
from IPython.display import Markdown

comparison = json.loads((FINAL_COMPARISON / 'comparison.json').read_text())
heatmaps = json.loads(
    (HEATMAP_COMPARISON / 'cka_before_after_summary.json').read_text()
) if RUN_HEATMAPS else {'modules': {}}
rows = {row['metric']: row for row in comparison['comparison']}
def change(metric):
    value = rows.get(metric, {}).get('change_percent')
    return 'N/A' if value is None else f'{value:+.2f}%'

validity = (
    'ĐỦ ĐIỀU KIỆN đánh giá offline có kiểm soát'
    if EXPERIMENT_MODE == 'full' and DATASETS_ARE_DISJOINT
    else 'CHỈ LÀ SMOKE/DEBUG — không dùng làm kết luận nghiên cứu'
)
heatmap_lines = []
for name, values in heatmaps.get('modules', {}).items():
    heatmap_lines.append(
        '- `{}`: |ΔCKA| trung bình {:.5f}, Frobenius {:.5f}.'.format(
            name, values['delta_mean_absolute'], values['delta_frobenius']
        )
    )
display(Markdown(f'''
# Kết luận

**Trạng thái thí nghiệm:** {validity}

- Parameters: {change('parameter_count')}
- Mean latency: {change('latency_mean_ms')}
- P95 latency: {change('latency_p95_ms')}
- Peak allocated VRAM: {change('peak_allocated_gib')}
- MSE: {change('mse')}
- MAE: {change('mae')}
- Inference speedup: {comparison['inference_speedup']:.3f}×

## Thay đổi biểu diễn nội bộ
{chr(10).join(heatmap_lines) if heatmap_lines else '- Không chạy heatmap.'}

MSE/MAE và heatmap chỉ là đánh giá offline. Muốn tuyên bố tương đương CLP cần 
bổ sung simulator/robot rollout success rate, nhiều seed và confidence interval.
'''))


In [ ]:
# Đóng gói toàn bộ kết quả để tải xuống.
archive = WORK_ROOT / 'gr00t_n1d7_cka_results'
run_checked('Archive results', [
    'bash', '-lc', f"cd '{OUTPUT_ROOT.parent}' && zip -qr '{archive}.zip' '{OUTPUT_ROOT.name}'"
])
print(f'Kết quả: {archive}.zip')
